# ทดลอง CLA/CNN จากแผ่นที่ตรวจคำตอบครบแล้ว

ชุดสุดท้ายใช้ 270 คู่ / 540 ภาพ และไม่มีการเก็บภาพเพิ่ม รอบแรกเลือกเฉพาะแผ่นที่คำตอบจริงผ่านการตรวจครบ 100 ข้อ (ปัจจุบัน 38 แผ่น / 76 ภาพ / 3,800 คำตอบ) คำตอบที่ตรวจแล้วอีก 960 ข้อยังอยู่ในไฟล์หลัก ไม่ใช้ร่าง OMR เป็นเฉลย

อัปโหลด `omr_reviewed_pilot.zip` เป็น Kaggle Dataset แบบ Private แล้วแนบให้โน้ตบุ๊ก ไฟล์ ZIP รวมภาพชุดทดลอง คำตอบที่ตรวจแล้ว และ `src/omr_pipeline.py` ไว้แล้ว

**ผลรอบนี้เป็นการทดลองพัฒนาระบบจากป้ายกำกับที่มี OMR ช่วยตรวจ ไม่ใช่ความแม่นยำจากการทดสอบอิสระ** ชุดย่อยยังไม่มีตัวอย่างคลาสฝนหลายช่องหรือดูไม่ชัดครบพอ และคงการแบ่ง Train/Validation/Test เดิมตามแผ่น


In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import shutil

WORKING = Path('/kaggle/working')

INPUT = Path('/kaggle/input')
DATA_ROOTS = [p.parent.parent.parent for p in INPUT.rglob('image_inventory_verified.csv') if p.parent.name == 'metadata']
CODE_SCRIPTS = list(INPUT.rglob('omr_pipeline.py'))

if not DATA_ROOTS:
    raise FileNotFoundError('แนบ Kaggle Dataset ที่มี data/metadata/image_inventory_verified.csv และ Dataset-Computer-vision/')
if not CODE_SCRIPTS:
    raise FileNotFoundError('แนบ Kaggle Dataset โค้ดที่มี src/omr_pipeline.py')

DATA_ROOT = DATA_ROOTS[0]
PIPELINE = CODE_SCRIPTS[0]
META = DATA_ROOT / 'data' / 'metadata'
for required in ['answer_labels.csv', 'sheet_splits.csv']:
    if not (META / required).is_file():
        raise FileNotFoundError(f'ยังไม่มี {META / required}; จับคู่ภาพและกรอก Ground truth ก่อน')

print('Data root:', DATA_ROOT)
print('Pipeline:', PIPELINE)
print('GPU available:', bool(shutil.which('nvidia-smi')) and subprocess.run(['nvidia-smi'], capture_output=True).returncode == 0)


## เตรียมข้อมูลที่คนตรวจครบสำหรับทดลอง

เลือกเฉพาะแผ่นที่มี A/B ครบและคำตอบจริงครบ 100 ข้อ ตรวจว่าไม่มีคู่ติดพัก เก็บแผ่นและ split เดิม ไม่เติมร่างเป็นคำตอบจริง และเขียนข้อมูลชุดย่อยลง `/kaggle/working` โดยไม่แก้ไฟล์ต้นฉบับ


In [ ]:
import pandas as pd
from IPython.display import display

inventory = pd.read_csv(META / 'image_inventory_verified.csv', dtype=str, keep_default_na=False)
labels = pd.read_csv(META / 'answer_labels.csv', dtype=str, keep_default_na=False)
splits = pd.read_csv(META / 'sheet_splits.csv', dtype=str, keep_default_na=False)
verified = inventory[inventory.pair_status.eq('verified')].copy()
scope_file = META / 'dataset_scope.json'
scope = json.loads(scope_file.read_text(encoding='utf-8')) if scope_file.exists() else {'final_paired_sheets': 270}
if scope['final_paired_sheets'] != 270 or verified.sheet_id.nunique() != scope.get('input_subset_sheets', 270):
    raise ValueError('ขนาดข้อมูลไม่ตรงกับขอบเขตชุดหลัก 270 คู่ หรือชุดย่อยที่จัดเตรียมไว้')
valid_marks = {'1', '2', '3', '4', '5', 'blank', 'multiple', 'ambiguous'}
if labels.duplicated(['sheet_id', 'question_no']).any():
    raise ValueError('พบคำตอบซ้ำใน sheet_id/question_no')
complete = {sid for sid, rows in labels.groupby('sheet_id')
            if len(rows) == 100 and set(rows.question_no) == {str(q) for q in range(1, 101)}
            and rows.ground_truth.isin(valid_marks).all()}
roles = verified.groupby('sheet_id').capture_role.agg(lambda x: sorted(x.tolist()))
complete &= {sid for sid, pair in roles.items() if pair == ['A', 'B']}
hold_file = META / 'pairing_review_holds.csv'
if hold_file.exists():
    holds = pd.read_csv(hold_file, dtype=str, keep_default_na=False)
    complete -= set(holds.loc[holds.review_status.eq('pending'), 'sheet_id'])
print('Official inspected labels in source:', labels.ground_truth.ne('').sum())
inventory = verified[verified.sheet_id.isin(complete)].copy()
labels = labels[labels.sheet_id.isin(complete)].copy()
splits = splits[splits.sheet_id.isin(complete)].copy()
if set(splits.split) != {'train', 'validation', 'test'} or set(splits.sheet_id) != complete:
    raise ValueError('ชุดทดลองต้องมีแผ่นและ split เดิมครบทุกกลุ่ม')
PILOT_META = WORKING / 'reviewed-pilot-metadata'
PILOT_META.mkdir(parents=True, exist_ok=True)
for name, frame in [('image_inventory_verified.csv', inventory), ('answer_labels.csv', labels), ('sheet_splits.csv', splits)]:
    frame.to_csv(PILOT_META / name, index=False)
print('Pilot sheets:', len(complete), 'images:', len(inventory), 'human labels:', len(labels))
print('Drafts used as labels: 0')
display(splits.groupby('split').sheet_id.nunique().rename('sheets').to_frame())


In [ ]:
OUTPUT = WORKING / 'omr-reviewed-pilot-results'
command = [
    sys.executable, str(PIPELINE),
    '--data-root', str(DATA_ROOT),
    '--inventory', str(PILOT_META / 'image_inventory_verified.csv'),
    '--labels', str(PILOT_META / 'answer_labels.csv'),
    '--splits', str(PILOT_META / 'sheet_splits.csv'),
    '--heic-map', 'data/metadata/heic_conversion_map.csv',
    '--output-dir', str(OUTPUT),
    '--epochs', '12',
    '--seed', '20261025',
]
subprocess.run(command, check=True)

report_path = OUTPUT / 'evaluation.json'
report = json.loads(report_path.read_text(encoding='utf-8'))
report.update(annotation_mode='omr_assisted', evaluation_type='assisted_development_pilot', independent_evaluation=False,
              final_dataset_pairs=270, pilot_sheets=len(complete), pilot_human_labels=len(labels), omr_drafts_used_as_labels=0)
report_path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')


In [ ]:
report = json.loads((OUTPUT / 'evaluation.json').read_text(encoding='utf-8'))
print('Evaluation:', report['evaluation_type'], 'independent:', report['independent_evaluation'])
print('Split by sheet:', report['split_sheets'])
print('CLA thresholds:', report['cla_fill_threshold'], report['cla_margin_threshold'])
print('CNN confidence cutoff:', report['cnn_validation_confidence_cutoff'])
for model_name, values in report['models'].items():
    print('\n', model_name.upper())
    for workflow in ('A_only', 'A_then_B'):
        metrics = values[workflow]
        print(workflow, 'accuracy=', round(metrics['accuracy'], 4), 'macro F1=', round(metrics['macro_f1'], 4), 'coverage=', round(metrics['coverage'], 4))


## ตัวอย่างเดโมจาก split Test ของชุดทดลอง

ดูภาพ A/B และผลอ่านของ CLA/CNN ในแผ่นเดียวกัน ผลนี้ใช้ตรวจการทำงานของระบบ และยังเป็นการทดลองจากป้ายกำกับที่มี OMR ช่วยตรวจ


In [ ]:
from IPython.display import display, Image as DisplayImage

test_sheet = splits.loc[splits.split.eq('test'), 'sheet_id'].iloc[0]
pair = inventory[(inventory.sheet_id == test_sheet) & (inventory.pair_status == 'verified')].set_index('capture_role')
conversion_map = {}
map_file = META / 'heic_conversion_map.csv'
if map_file.exists():
    conversion_df = pd.read_csv(map_file)
    conversion_map = dict(zip(conversion_df.source_relative_path, conversion_df.normalized_relative_path))
for role in ['A', 'B']:
    rel = pair.loc[role, 'relative_path']
    img_path = DATA_ROOT / rel
    if img_path.suffix.lower() in {'.heic', '.heif'}:
        normalized = conversion_map.get(Path(rel).as_posix())
        if not normalized:
            raise FileNotFoundError(f'ไม่มี JPEG conversion สำหรับ {rel}')
        img_path = DATA_ROOT / normalized
    print(test_sheet, role)
    display(DisplayImage(filename=str(img_path), width=360))

cnn_demo = pd.read_csv(OUTPUT / 'cnn_test_predictions.csv')
cla_demo = pd.read_csv(OUTPUT / 'cla_test_predictions.csv')
demo = cnn_demo[cnn_demo.sheet_id.eq(test_sheet)].merge(
    cla_demo[['sheet_id', 'question_no', 'a_only', 'a_then_b']],
    on=['sheet_id', 'question_no'], suffixes=('_cnn', '_cla')
)
display(demo[['question_no', 'ground_truth', 'a_only_cla', 'a_then_b_cla', 'a_only_cnn', 'a_then_b_cnn', 'used_B']].sort_values('question_no'))
